# Fine-tuning de Gemma 4 E4B (LoRA / Unsloth) avec descriptions MedGemma

On fine-tune `google/gemma-4-E4B-it` (miroir 4-bit Unsloth) en utilisant le CSV de descriptions
généré au notebook 1. Le modèle apprend à produire un **JSON** :

```json
{"predicted_class": "normal" | "suspected_opacity",
 "opacities": [{"box_2d": [y1, x1, y2, x2]}],
 "description": "..."}
```

Le format `box_2d [y1, x1, y2, x2]` normalisé **0-1024** est identique à celui de ton notebook
d'inférence, pour que le notebook 3 réutilise tes fonctions de dessin directement.

**Prérequis :** GPU (P100 ou T4×2), licence Gemma acceptée, `HF_TOKEN` dans les Secrets,
et le CSV `pneumonia_descriptions.csv` ajouté en dataset d'entrée.

In [1]:
# === 1. Installation (Unsloth) — on ne touche PAS à Pillow ===
!pip install -q --upgrade unsloth unsloth_zoo
!pip install -q pydicom opencv-python
# Sécurité : réaligne Pillow sur la version de l'image Kaggle si besoin
!pip install -q --force-reinstall --no-deps "pillow==11.3.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.0/74.0 MB 25.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 49.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 99.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 112.4 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 99.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 84.8 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━

In [3]:
# === 2. Auth Hugging Face ===
import os
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN
login(HF_TOKEN)
print("logged in")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


logged in


In [4]:
# === 3. Configuration ===
import glob

if os.path.exists('/kaggle/input/competitions/rsna-pneumonia-detection-challenge'):
    BASE_DIR = '/kaggle/input/competitions/rsna-pneumonia-detection-challenge/'
else:
    BASE_DIR = '.'
TRAIN_IMAGES_DIR = os.path.join(BASE_DIR, 'stage_2_train_images')

# CSV de descriptions du notebook 1 : auto-détection dans /kaggle/input/**
cands = glob.glob('/kaggle/input/**/pneumonia_descriptions.csv', recursive=True)
DESC_CSV = cands[0] if cands else '/kaggle/input/datasets/jeankevin56/pneumonia-full-csv/pneumonia_descriptions.csv'
print("CSV descriptions :", DESC_CSV)

MODEL_NAME  = "unsloth/gemma-4-E4B-it"
IMAGE_SIZE  = 512
SOURCE_SIZE = 1024
MAX_STEPS   = 60          # mets None + num_train_epochs pour un run complet
OUTPUT_DIR  = "/kaggle/working/gemma4_pneumonia_lora"
SEED        = 3407

CSV descriptions : /kaggle/input/datasets/jeankevin56/pneumonia-full-csv/pneumonia_descriptions.csv


In [5]:
# === 4. Imports image + DICOM ===
import json, numpy as np, pandas as pd
import pydicom, cv2
from PIL import Image

In [6]:
def process_dicom(patient_id, resize=None):
    """Lit un DICOM, normalise (Min-Max), applique CLAHE, renvoie une image PIL RGB.
    Si resize est donné (ex. 512), renvoie une image carrée redimensionnée.
    """
    path = os.path.join(TRAIN_IMAGES_DIR, f"{patient_id}.dcm")
    if not os.path.exists(path):
        return None
    dcm = pydicom.dcmread(path)
    img = dcm.pixel_array
    img = (img - np.min(img)) / (np.max(img) - np.min(img) + 1e-8) * 255.0
    img = img.astype(np.uint8)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    img = clahe.apply(img)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
    pil = Image.fromarray(img_rgb)
    if resize is not None:
        pil = pil.resize((resize, resize))
    return pil

In [7]:
# === 5. Lecture du CSV + construction des cibles JSON ===
desc_df = pd.read_csv(DESC_CSV)
print("Exemples :", len(desc_df))

def xywh_to_box2d(box):
    """[x, y, w, h] px d'origine -> [y1, x1, y2, x2] normalisé 0-1024 (format box_2d)."""
    x, y, w, h = box
    y1 = round(y / SOURCE_SIZE * 1024); x1 = round(x / SOURCE_SIZE * 1024)
    y2 = round((y + h) / SOURCE_SIZE * 1024); x2 = round((x + w) / SOURCE_SIZE * 1024)
    return [y1, x1, y2, x2]

def build_target_json(row):
    boxes = json.loads(row['boxes']) if isinstance(row['boxes'], str) else (row['boxes'] or [])
    if len(boxes) > 0:
        opac = [{"box_2d": xywh_to_box2d(b)} for b in boxes]
        out = {"predicted_class": "suspected_opacity", "opacities": opac, "description": row['description']}
    else:
        out = {"predicted_class": "normal", "opacities": [], "description": row['description']}
    return json.dumps(out, ensure_ascii=False)

Exemples : 500


In [8]:
# === 6. Dataset conversationnel (image AVANT le texte) ===
from tqdm.auto import tqdm

INSTRUCTION = (
    """
You are an AI assistant designed for educational purposes only.
Analyze this frontal chest X-ray for signs of pneumonia.

Classify the image as one of:
- "suspected_opacity": one or more focal opacities/consolidations consistent with pneumonia are visible
- "normal": no such opacity is visible
- "uncertain": you cannot confidently choose between the two
If predicted_class is "suspected_opacity", report 1 or 2 opacities — the most
clinically significant ones. If multiple affected regions are adjacent or overlapping,
merge them into a single bounding box covering the overall extent rather than listing
many small boxes.

If predicted_class is "normal" or "uncertain", "opacities" must be an empty list [].
Answer ONLY with a JSON object:
{
"predicted_class": "normal" or "suspected_opacity",
"opacities": [{"box_2d": [y1, x1, y2, x2]}] (coordinates normalised to 0-1024, empty list if normal),
"description": a short clinical description
}
    """
)

def to_conversation(row):
    img = process_dicom(row['patientId'], resize=IMAGE_SIZE)
    target = build_target_json(row)
    return {"messages": [
        {"role": "user", "content": [
            {"type": "image", "image": img},
            {"type": "text",  "text": INSTRUCTION},
        ]},
        {"role": "assistant", "content": [{"type": "text", "text": target}]},
    ]}

dataset = []
for _, r in tqdm(desc_df.iterrows(), total=len(desc_df)):
    img_path = os.path.join(TRAIN_IMAGES_DIR, f"{r['patientId']}.dcm")
    if os.path.exists(img_path):
        dataset.append(to_conversation(r))
print("Dataset :", len(dataset))
print("Cible exemple :", dataset[0]["messages"][1]["content"][0]["text"])

  0%|          | 0/500 [00:00<?, ?it/s]

Dataset : 500
Cible exemple : {"predicted_class": "normal", "opacities": [], "description": "This is a portable AP chest X-ray. The lungs appear clear with normal lung markings. There are no areas of opacity or consolidation visible. The costophrenic angles are sharp. The heart size is normal. The mediastinum is centered. There are no signs of pleural effusion. The image is consistent with a normal chest X-ray."}


In [10]:
# === 7. Chargement de Gemma 4 E4B en 4-bit ===
from unsloth import FastVisionModel
import torch

model, processor = FastVisionModel.from_pretrained(
    MODEL_NAME,
    load_in_4bit = True,
    use_gradient_checkpointing = "unsloth",
)

==((====))==  Unsloth 2026.6.9: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/2130 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/203 [00:00<?, ?B/s]

processor_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

In [11]:
# === 8. Adaptateurs LoRA ===
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = True,  # True si >20 Go VRAM (meilleure localisation)
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r = 16, lora_alpha = 16, lora_dropout = 0, bias = "none",
    random_state = SEED, target_modules = "all-linear",
)

In [12]:
# === 9. Template de chat Gemma 4 ===
from unsloth import get_chat_template
processor = get_chat_template(processor, "gemma-4")

In [13]:
# === 10. Entraînement (loss 13-15 = normal pour ces modèles multimodaux) ===
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

cfg = SFTConfig(
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 4,
    warmup_ratio = 0.03,
    max_steps = MAX_STEPS,         # -> None + num_train_epochs pour run complet
    # num_train_epochs = 1,
    learning_rate = 2e-4,
    logging_steps = 1,
    optim = "adamw_8bit",
    weight_decay = 0.001,
    lr_scheduler_type = "cosine",
    max_grad_norm = 0.3,
    save_strategy = "no",
    seed = SEED, output_dir = "outputs", report_to = "none",
    remove_unused_columns = False,
    dataset_text_field = "",
    dataset_kwargs = {"skip_prepare_dataset": True},
    max_length = 1024,
)
trainer = SFTTrainer(
    model = model, train_dataset = dataset,
    processing_class = processor.tokenizer,
    data_collator = UnslothVisionDataCollator(model, processor),
    args = cfg,
)
trainer_stats = trainer.train()

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Model does not have a default image size - using 512


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 500 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 41,222,144 of 8,037,378,592 (0.51% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
1,0.738569
2,0.754059
3,0.694900
4,0.578956
5,0.463980
6,0.393299
7,0.331029
8,0.275371
9,0.238732
10,0.190537


In [14]:
# === 11. Sauvegarde + archive téléchargeable + (option) push HF ===
import shutil
model.save_pretrained(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
shutil.make_archive("/kaggle/working/gemma4_pneumonia_lora", "zip", OUTPUT_DIR)
print("Adaptateur sauvegardé + zippé :", OUTPUT_DIR + ".zip")

Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/gemma4_pneumonia_lora/tokenizer_config.json.


Adaptateur sauvegardé + zippé : /kaggle/working/gemma4_pneumonia_lora.zip


In [15]:
model.push_to_hub("JeanKevin56/gemma4-pneumonia-lora", token=HF_TOKEN)
processor.push_to_hub("JeanKevin56/gemma4-pneumonia-lora", token=HF_TOKEN)

README.md:   0%|          | 0.00/573 [00:00<?, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Saved model to https://huggingface.co/JeanKevin56/gemma4-pneumonia-lora


Unsloth: Restored added_tokens_decoder metadata in /tmp/tmp80p1ll18/tokenizer_config.json.


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

## Étape suivante
- **Soit** tu pousses l'adaptateur sur HF (décommente la cellule 11) → le notebook 3 le charge par son nom de repo.
- **Soit** tu fais *Save Version*, puis dans le notebook 3 tu ajoutes la sortie de ce notebook comme **dataset d'entrée** (chemin `/kaggle/input/...`).